Create an orchestrator abstraction that forwards a prompt to various LLM's parallely and gathers the results.
This abstraction can be used to get response to a question first, followed by a judgement ranking from all the LLM's.
The final ranking is an average of the ranking from all the judgment respones.

We will use asyncio to achieve the parallelism.

We will also use the async client SDK instead of the regular client to achieve parallelism.

Prerequisites: The code expects the keys - "OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GOOGLE_API_KEY", "GROQ_API_KEY", "DEEPSEEK_API_KEY" and "OLLAMA_API_KEY", to be set in the .env file.

Caveats: The models[] list gets appended to every time the prompt_forwarder is called. So it will grow beyond what is required, but it works correctly. Good enough to demonstrate the underlying experiment, not for memory efficiency. 

In [ ]:
# Boiler plate code
from dotenv import load_dotenv
import os
import asyncio
import json
from collections import defaultdict
from openai import OpenAI, AsyncOpenAI
from anthropic import Anthropic, AsyncAnthropic

load_dotenv()

openai_key = os.getenv("OPENAI_API_KEY")
anthropic_key = os.getenv("ANTHROPIC_API_KEY")
google_key = os.getenv("GOOGLE_API_KEY")
groq_key = os.getenv("GROQ_API_KEY")
deepseek_key = os.getenv("DEEPSEEK_API_KEY")
ollama_key = os.getenv("OLLAMA_API_KEY")

if not openai_key:
    print("OpenAI API key not found.")
if not anthropic_key:
    print("Anthropic key not found.")
if not google_key:
    print("Google key not found.")
if not groq_key:
    print("Groq key not found.")
if not deepseek_key:
    print("Deepseek key not found.")
if not ollama_key:
    print("Ollama key not found.")

GOOGLE_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GROQ_BASE_URL = "https://api.groq.com/openai/v1"
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
OLLAMA_BASE_URL = "http://localhost:11434/v1"

In [ ]:
# Generate a question
message = "Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. Answer only with the question, no explanation."
openai_client = OpenAI()
response = openai_client.chat.completions.create(
    model = "gpt-5-nano",
    messages = [
        {"role": "user", "content": message}
    ]
)
question = response.choices[0].message.content
print(question)

In [ ]:
# Create an orchestrator using asyncio that forwards the prompt to multiple LLM's parallely and gathers the results.
# To keep it simple, we will fail the entire operation if any of the calls fail.
models = []
async def call_openai(prompt: str) -> str:
    openai_client = AsyncOpenAI()
    model = "gpt-5.4-nano"
    models.append(model)
    response = await openai_client.chat.completions.create(
        model = model,
        messages = [
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

async def call_anthropic(prompt: str) -> str:
    anthropic_client = AsyncAnthropic()
    model = "claude-haiku-4-5"
    models.append(model)
    response = await anthropic_client.messages.create(
        model = model,
        messages = [
            {"role": "user", "content": prompt}
        ],
        max_tokens = 2000
    )
    return response.content[0].text

async def call_google(prompt: str) -> str:
    google_client = AsyncOpenAI(api_key = google_key, base_url = GOOGLE_BASE_URL)
    model = "gemini-2.5-flash"
    models.append(model)
    response = await google_client.chat.completions.create(
        model = model,
        messages = [
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

async def call_groq(prompt: str) -> str:
    groq_client = AsyncOpenAI(api_key = groq_key, base_url = GROQ_BASE_URL)
    model = "llama-3.3-70b-versatile"
    models.append(model)
    response = await groq_client.chat.completions.create(
        model = model,
        messages = [
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

async def call_deepseek(prompt: str) -> str:
    deepseek_client = AsyncOpenAI(api_key = deepseek_key, base_url = DEEPSEEK_BASE_URL)
    model = "deepseek-v4-flash"
    models.append(model)
    response = await deepseek_client.chat.completions.create(
        model = model,
        messages = [
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

async def call_ollama(prompt: str) -> str:
    ollama_client = AsyncOpenAI(api_key = ollama_key, base_url = OLLAMA_BASE_URL)
    model = "glm-4.7-flash:latest"
    models.append(model)
    response = await ollama_client.chat.completions.create(
        model = model,
        messages = [
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

async def prompt_forwarder(prompt: str) -> list[str]:
    # comment out tasks to toggle
    tasks = [
        call_openai(prompt),
        call_anthropic(prompt),
        call_google(prompt),
        call_groq(prompt),
        call_deepseek(prompt),
        # call_ollama(prompt)
    ]
    results = await asyncio.gather(*tasks, return_exceptions=False)
    return results

answers = await prompt_forwarder(question)
i = 1
for answer in answers:
    print(f"-----RESPONSE {i} START------")
    i += 1
    print(answer, end="\n\n")

In [ ]:
# Judge the responses by getting the ranking from all of the models. Reuse the prompt_forwarder().
# Build a single prompt from all the responses.
# Let's bring this together - note the use of "enumerate"

together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index+1}\n\n"
    together += answer + "\n\n"

judge = f"""You are judging a competition between {len(answers)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with correct JSON, and only JSON, with the following format:
["best competitor number", "second best competitor number", "third best competitor number", ...]
Example output: ["2", "1", "5", "6", "3", "4"]

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks.
"""

rankings = await prompt_forwarder(judge)
print(rankings)

In [ ]:
# Create a cummulative rank by calculating the average rank of each

cummulative_rank = defaultdict[int, int](int)

for ranking in rankings:
    ranks = json.loads(ranking)
    print(ranks)
    for idx, model_num in enumerate(ranks, start = 1):
        cummulative_rank[int(model_num)] += idx
    print(cummulative_rank)

print("=== FINAL RANKING OF ANSWERS ===")
ranking = sorted(cummulative_rank, key = cummulative_rank.get)
print(ranking)
print(models)
for idx, model in enumerate(ranking):
    # models[] is 0 indexed, so subtract model by 1
    print(f"RANK {idx+1}: {models[model - 1]}")